In [ ]:
#update on 27/08/2025 @ 3:00 pm

# by JyW



# maintenance done on 27/06/2022 @ 3:05 pm

# by Gabin



#@author: PristerM

#update by LLZ

#on 21/02/2022





# %%



#------------------------------------------------ Begin_Librairie ----------------------------------------



from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager

from time import sleep

import os

import re

import requests

import pdfplumber

import xml.etree.ElementTree as ET

# import camelot





# %%



#------------------------------------------------ Begin_ fileName ----------------------------------------



print("Running MY LFSA Web Scraping Tool v.1.0")





regulatorName = 'MY LFSA' ## change to current controller name





now=datetime.datetime.now()



filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"



scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment



os.chdir(scriptfolder)



writer = ExcelWriter(filename)



tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process







if os.path.exists(tempfolder):



    for rem in os.listdir(tempfolder):



        os.remove(os.path.join(tempfolder, rem))



else:



    os.mkdir(tempfolder)



# %%



#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder



chromeOptions = webdriver.ChromeOptions()



prefs = {"plugins.always_open_pdf_externally": True,



		 "download.prompt_for_download": False,



		 "download.default_directory" : tempfolder}



chromeOptions.add_experimental_option("prefs",prefs)



driver = webdriver.Chrome(options=chromeOptions)



driver.maximize_window()









    

# %%



#------------------------------------------------ Begin_Variable ----------------------------------------



regdict={

    'MY LFSA 1': 'https://www.labuanfsa.gov.my/areas-of-business/financial-services/banking/list-of-labuan-banks-and-investment-banks',

    'MY LFSA 2':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/banking/list-of-labuan-banks-and-investment-banks',

    'MY LFSA 3':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/insurance/list-of-labuan-insurance-insurance-related-entities',

    'MY LFSA 4':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/insurance/list-of-labuan-insurance-insurance-related-entities',

    'MY LFSA 5':'https://www.labuanfsa.gov.my/areas-of-business/labuan-service-providers/trust-companies-and-ancillary-services/list-of-labuan-trust-companies',

    'MY LFSA 6':'https://www.labuanfsa.gov.my/areas-of-business/labuan-service-providers/trust-companies-and-ancillary-services/list-of-labuan-trust-companies',

    #       'MY LFSA 7':'https://www.labuanibfc.com/areas-of-business/financial-services/leasing/list-of-leasing-companies',

    'MY LFSA 7':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/leasing/list-of-leasing-companies',

    'MY LFSA 8':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/leasing/list-of-leasing-companies',

    'MY LFSA 9':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/commodity-trading/list-of-labuan-international-commodity-trading-companies',

    'MY LFSA 10':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/capital-markets/list-of-fund-managers'

         }



Typology={

'MY LFSA 1':	'List of Labuan Banks and Investment Banks',

'MY LFSA 2':	'List of Labuan Banks & Investment Bank - Surrendered & Revoked Licence',

'MY LFSA 3':    'List of Insurance and Insurance Related Companies',

'MY LFSA 4':	'List of Labuan Insurance & Insurance Related - Surrendered & Revoked Licence',

'MY LFSA 5':	'List of Labuan Trust Companies',

'MY LFSA 6':	'List of Labuan Trust Companies - Surrendered & Revoked Licence',

'MY LFSA 7':	'List of Leasing Companies in Labuan',

'MY LFSA 8':	'List of Leasing Companies - Ceased Operations',

'MY LFSA 9':	'List of Labuan International Commodity Trading Companies',

'MY LFSA 10':	'List of Labuan Fund Managers',}







sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 



	  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 



		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 



		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],



		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 



		  'Phone - Mother company': [], 'Check': []}





try:

    os.mkdir(tempfolder)

except:

    prevfiles=os.listdir(tempfolder)

    os.chdir(tempfolder)

    for prf in prevfiles:

        os.remove(prf)

    print('The directory tempfolder already exists.')

os.chdir(tempfolder)##only if files are going to be downloaded here



processdate=now.strftime('%Y-%m-%d')

pattern = re.compile('([0-9]+)')





# %%



#------------------------------------------------ Begin_Fouction ----------------------------------------



def bourange_same_length_array(sqldict) :



    len_value=[]



    for key, value in sqldict.items():



        len_value.append(len(value))



    maxlen = max(len_value)



    for key, val in sqldict.items():



        if len(sqldict[key]) != maxlen:



            empty = []



            total_empty = maxlen - len(sqldict[key])



            for i in range(total_empty):



                empty.append('')



            sqldict[key]=sqldict[key]+empty



    return sqldict



# %%



#------------------------------------------------ Begin_Main ----------------------------------------



for reg in regdict:

    print('Working with {}'.format(reg))

    driver.get(regdict[reg])

    sleep(3)

    

    if reg in ['MY LFSA 2', 'MY LFSA 4', 'MY LFSA 6', 'MY LFSA 8']:

        

        soup=BeautifulSoup(driver.page_source,"html.parser")

        div = soup.find('div',{'class':'content list-container list-d-gap-large list-t-gap-medium list-m-gap-small list-d-col-1 list-t-col-1 list-m-col-1'})

        a_s = div.find_all('a')

        for a in a_s:

            if 'Surrendered & Revoked'  in a.text.strip() or 'Ceased' in a.text.strip():

                #print(a.text.strip())

                sleep(2)

                driver.get(a['href'])

                

                while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:

                    print('Waiting for file to download')

                    sleep(2)

    

                pdf_file = os.listdir(tempfolder)[0]

                filePath = os.path.join(tempfolder, pdf_file)

                if reg not in [ 'MY LFSA 8' ,'MY LFSA 6']:

                    types_ = ''

                    with pdfplumber.open(filePath) as pdf:

                        for page in pdf.pages:

                            #print(page)

                            # Iterate over tables and print them

                            tables = page.extract_tables()

                            for i, table in enumerate(tables):

                                #print(f"Table {i + 1}:")

                                

                                for row in table:

                                    #print(row)

                                    try:

                                        if row[0].isdigit():

                                            sqldict['Name'].append(row[1])

                                            sqldict['License_Type'].append(row[2])

                                            sqldict["CancellationDate"].append(row[3])

                                            sqldict["ListProcessDate"].append(processdate)

                                            sqldict["RegCtry"].append("MY")    

                                            sqldict["RegCode"].append("LFSA")

                                            sqldict['ListCode'].append(reg.split(' ')[-1])

                                            sqldict["Cntry"].append("MY")

                                            sqldict['ListName'].append(Typology[reg])

                                        else:



                                            for index, info in enumerate(row):

                                                # Normalize the string for easier matching

                                                normalized_info = info.lower()



                                                # Check for date-related keywords with revoked or surrendered

                                                if re.search(r'\b(date\s*(of)?\s*(revoked|surrendered)|revoked\s*date|surrendered\s*date|date\s*revoked|date\s*surrendered)\b', normalized_info):

                                                    #print(f"Index: {index}")

                                                    

                                                    # Determine the type based on keywords

                                                    if 'surrender' in normalized_info:

                                                        types_ = 'Surrendered'

                                                    elif 'revoked' in normalized_info:

                                                        types_ = 'Revoked'

                                                    else:

                                                        types_ = ''



                                            print(f"Type: {types_}")

                                        sqldict["RegulationType"].append(types_)

                                        sqldict = bourange_same_length_array(sqldict)



                                    except:

                                        pass

                elif reg == 'MY LFSA 8':

                    with pdfplumber.open(filePath) as pdf:

                        for page in pdf.pages:

                            #print(page)

                            # Iterate over tables and print them

                            tables = page.extract_tables()

                            for i, table in enumerate(tables):

                                #print(f"Table {i + 1}:")

                                

                                for row in table:

                                    #print(row)

                                    try:

                                        if row[0].isdigit():

                                            #print(row)

                                            

                                            sqldict['Name'].append(row[1])

                                            sqldict["CancellationDate"].append(row[2])

                                            sqldict["ListProcessDate"].append(processdate)

                                            sqldict["RegCtry"].append("MY")    

                                            sqldict["RegCode"].append("LFSA")

                                            sqldict['ListCode'].append(reg.split(' ')[-1])

                                            sqldict["Cntry"].append("MY")

                                            sqldict['ListName'].append(Typology[reg])

                                        #print(f"Type: {types_}")

                                        sqldict["RegulationType"].append('Ceased')

                                        sqldict = bourange_same_length_array(sqldict)



                                    except:

                                        pass

                elif reg == 'MY LFSA 6':

                    types_ = ''

                    with pdfplumber.open(filePath) as pdf:

                        for page in pdf.pages:

                            #print(page)

                            # Iterate over tables and print them

                            tables = page.extract_tables()

                            for i, table in enumerate(tables):

                                #print(f"Table {i + 1}:")

                                for row in table:

                                    #print(row)

                                    try:

                                        if row[0].isdigit():

                                            sqldict['Name'].append(row[1])

                                            sqldict['License_Type'].append(row[2])

                                            sqldict["CancellationDate"].append(row[3])

                                            sqldict["ListProcessDate"].append(processdate)

                                            sqldict["RegCtry"].append("MY")    

                                            sqldict["RegCode"].append("LFSA")

                                            sqldict['ListCode'].append(reg.split(' ')[-1])

                                            sqldict["Cntry"].append("MY")

                                            sqldict['ListName'].append(Typology[reg])

                                        else:



                                            if any('revoked' in cell.lower() for cell in row):

                                                types_ = 'Revoked'

                                            elif any('SURRENDER' in cell.lower() for cell in row):

                                                types_ = 'Surrendered'



                                            #print(f"Type: {types_}")

                                        sqldict["RegulationType"].append(types_)

                                        sqldict = bourange_same_length_array(sqldict)



                                    except:

                                        pass

                                        

        os.remove(filePath)           

    else:    

        soup = BeautifulSoup(driver.page_source, 'html.parser')



        # Find the <div> element containing the target text

        target_div = soup.find('div', string=lambda text: text and 'DOWNLOAD THIS FILE' in text)



        # Output the result

        if target_div:

            print("Found Button! " )

            driver.get(target_div.find_parent('a')['href'])

        else:

            print("No matching <div> found.")



        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:

            print('Waiting for file to download')

            sleep(2)



        excel_file = os.listdir(tempfolder)[0]

        filePath = os.path.join(tempfolder, excel_file)

        sleep(2)



        # Load and parse the XML

        tree = ET.parse(filePath)

        root = tree.getroot()



        # Set namespace used in XML

        ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}



        rows = []



        # Loop through each row in the XML

        for row in root.findall('.//ss:Row', ns):

            cells = []

            for cell in row.findall('ss:Cell', ns):

                data = cell.find('ss:Data', ns)

                if data is not None:

                    cells.append(data.text)

                else:

                    cells.append(None)

            rows.append(cells)



        # Convert to DataFrame

        df = pd.DataFrame(rows)

        df.columns = df.iloc[0]

        df = df [1:]

        # Assign values from df to sqldict

        sqldict['Name'].extend(df['Company Name'].tolist())

        sqldict['Address_1'].extend( df['Address'].tolist())

        sqldict['Phone'].extend( df['Tel'].tolist())

        sqldict['Fax'].extend( df['Fax'].tolist())

        sqldict['Email'].extend( df['Email'].tolist())



        # Create a list of the same processdate, matching the number of rows

        sqldict['ListProcessDate'] .extend( [processdate] * len(df))

        sqldict["RegulationType"] .extend( ['Regulated'] * len(df))

        sqldict["RegCtry"] .extend( [reg.split(' ')[0]]* len(df) )

        sqldict["RegCode"].extend( [reg.split(' ')[1]]* len(df) )

        sqldict['ListCode'] .extend( [reg.split(' ')[-1]]* len(df))

        sqldict["Cntry"] .extend( [reg.split(' ')[0]]* len(df) )

        sqldict['ListName'] .extend( [Typology[reg]] * len(df))

        try:

            sqldict['License_Type'].extend(df['Type of License'].tolist())

        except:

            pass



        # Ensure all fields in sqldict are the same length

        sqldict = bourange_same_length_array(sqldict)



        os.remove(filePath)

        

# %%



#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------



os.chdir(scriptfolder)



df=pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df = df.replace('TBA', '', regex=True)

df = df.fillna('')

df = df.replace('N/A','',regex = True)

df['Zip'] = df['Address_1'].apply(

    lambda x: re.search(r'\d{5,}', str(x)).group() if pd.notna(x) and re.search(r'\d{5,}', str(x)) else '')


df.to_excel(writer, 'SQL Ready', index=False)
writer.save()



writer.close()



driver.quit()



sleep(3)




    
    
    
    